## 異常檢測：自動編碼器（ConvAutoEncoder）與孤立森林

- 目標：掌握無監督異常檢測（Anomaly Detection）的核心策略。學會利用 PyTorch 實作卷積自動編碼器（ConvAutoEncoder）並計算「重建誤差（Reconstruction Error）」，同時使用 Scikit-Learn 實作孤立森林（Isolation Forest）與單類別支持向量機（One-Class SVM）作為對照組，建構全方位的晶圓點位異常偵測系統。


### 1. 經典無監督模型：孤立森林 vs One-Class SVM

- 實作：對於晶圓測試的電性特徵（如頻寬、漏電流、電阻），我們可能完全沒有「異常標籤（Label）」。我們需要使用孤立森林與 One-Class SVM，直接從海量的正常資料中，抓出特徵分佈極端奇異的異常晶粒。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
import matplotlib.pyplot as plt

# 模擬產線數據：500 顆正常晶粒的電性指標 + 15 顆偶發的極端異常晶粒
np.random.seed(42)
normal_dies = np.random.normal(loc=[28.0, 0.05], scale=[0.5, 0.01], size=(500, 2))
anomaly_dies = np.random.uniform(low=[22.0, 0.3], high=[25.0, 0.8], size=(15, 2))

# 合併成統一特徵矩陣
X_all = np.vstack([normal_dies, anomaly_dies])
df_dies = pd.DataFrame(X_all, columns=["bandwidth_ghz", "leakage_current_ma"])

# 實作孤立森林 (Isolation Forest)
# contamination 代表預估的異常比例 (這裡設定約 3%)
iso_forest = IsolationForest(contamination=0.03, random_state=42)
df_dies["if_pred"] = iso_forest.fit_predict(X_all)  # -1 代表異常，1 代表正常

# 實作單類別支持向量機 (One-Class SVM)
oc_svm = OneClassSVM(nu=0.03, kernel="rbf", gamma="scale")
df_dies["ocsvm_pred"] = oc_svm.fit_predict(X_all)

print(f"孤立森林偵測到的異常數量: {(df_dies['if_pred'] == -1).sum()} 顆")
print(f"One-Class SVM 偵測到的異常數量: {(df_dies['ocsvm_pred'] == -1).sum()} 顆")

### 2. 卷積自動編碼器 (ConvAutoEncoder) 與重建誤差

- 實作：如果我們要偵測的是 2D 晶圓圖（Wafer Bin Map）上的局部像素異常，經典表格模型就無能為力了。自動編碼器的核心物理意義在於：「只用正常晶圓訓練模型，模型會學會如何完美重構正常影像；當輸入包含異常缺陷的晶圓時，模型重構不出來，進而產生極大的重建誤差（Reconstruction Error）。」


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# 定義卷積自動編碼器網路架構 (ConvAutoEncoder)
class ConvAutoEncoder(nn.Module):
    def __init__(self):
        super(ConvAutoEncoder, self).__init__()
        # 編碼器 (Encoder)：降維並提取晶圓拓撲特徵
        self.encoder = nn.Sequential(
            nn.Conv2d(
                1, 16, kernel_size=3, stride=2, padding=1
            ),  # [B, 1, 10, 10] -> [B, 16, 5, 5]
            nn.ReLU(),
            nn.Conv2d(
                16, 32, kernel_size=3, stride=2, padding=1
            ),  # [B, 16, 5, 5] -> [B, 32, 3, 3]
            nn.ReLU(),
        )
        # 解碼器 (Decoder)：將特徵還原回原始晶圓圖形狀
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                32, 16, kernel_size=3, stride=2, padding=1, output_padding=0
            ),  # -> [B, 16, 5, 5]
            nn.ReLU(),
            nn.ConvTranspose2d(
                16, 1, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # -> [B, 1, 10, 10]
            nn.Sigmoid(),  # 限制輸出在 0~1 之間
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


# --- 建立模擬 10x10 的正常晶圓圖與 1 片異常晶圓圖 (含有刮痕) ---
# PyTorch 張量形狀規範: [Batch_Size, Channels, Height, Width]
normal_wafers = torch.FloatTensor(
    np.random.normal(loc=0.1, scale=0.02, size=(100, 1, 10, 10))
).clamp(0, 1)

anomaly_wafer = normal_wafers[0].clone().unsqueeze(0)
anomaly_wafer[0, 0, 4:8, 4:8] = 0.9  # 模擬大面積群聚不良 (Cluster / Scratch)

# --- 訓練 AutoEncoder（僅使用正常晶圓資料） ---
model = ConvAutoEncoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(20):  # 快速訓練演示
    optimizer.zero_grad()
    outputs = model(normal_wafers)
    loss = criterion(outputs, normal_wafers)
    loss.backward()
    optimizer.step()

# --- 計算重建誤差 (Reconstruction Error) 判定異常 ---
model.eval()
with torch.no_grad():
    # 計算正常晶圓的重建誤差
    reconstructed_normal = model(normal_wafers[1:2])
    normal_error = criterion(reconstructed_normal, normal_wafers[1:2]).item()

    # 計算異常晶圓的重建誤差
    reconstructed_anomaly = model(anomaly_wafer)
    anomaly_error = criterion(reconstructed_anomaly, anomaly_wafer).item()

print("=" * 60)
print(f"正常晶圓的重建誤差 (MSE): {normal_error:.5f}")
print(f"異常晶圓的重建誤差 (MSE): {anomaly_error:.5f} -> 【顯著放大！】")
print("=" * 60)

### 3. 異常檢測結果視覺化對照

- 我們將孤立森林與 One-Class SVM 的二維點位偵測結果進行繪圖，這能完美對應您的專案架構。


In [ ]:
plt.figure(figsize=(10, 4.5))

# --- 繪製孤立森林結果 ---
plt.subplot(1, 2, 1)
plt.scatter(
    df_dies[df_dies["if_pred"] == 1]["bandwidth_ghz"],
    df_dies[df_dies["if_pred"] == 1]["leakage_current_ma"],
    c="blue",
    alpha=0.5,
    label="Normal",
)
plt.scatter(
    df_dies[df_dies["if_pred"] == -1]["bandwidth_ghz"],
    df_dies[df_dies["if_pred"] == -1]["leakage_current_ma"],
    c="red",
    marker="x",
    s=50,
    label="Anomaly",
)
plt.title("孤立森林 (Isolation Forest) 偵測結果")
plt.xlabel("Bandwidth (GHz)")
plt.ylabel("Leakage (mA)")
plt.legend()

# --- 繪製 One-Class SVM 結果 ---
plt.subplot(1, 2, 2)
plt.scatter(
    df_dies[df_dies["ocsvm_pred"] == 1]["bandwidth_ghz"],
    df_dies[df_dies["ocsvm_pred"] == 1]["leakage_current_ma"],
    c="green",
    alpha=0.5,
    label="Normal",
)
plt.scatter(
    df_dies[df_dies["ocsvm_pred"] == -1]["bandwidth_ghz"],
    df_dies[df_dies["ocsvm_pred"] == -1]["leakage_current_ma"],
    c="orange",
    marker="s",
    s=40,
    label="Anomaly",
)
plt.title("單類別支持向量機 (One-Class SVM) 偵測結果")
plt.xlabel("Bandwidth (GHz)")
plt.legend()

plt.tight_layout()
plt.show()

- 總結：在半導體晶圓測試中，真實的產品缺陷（如局部短路或刮痕）發生機率極低，也就是所謂的『數據極度不平衡』。此時，若採用傳統的分類型監督學習，效果往往很差，因為模型根本沒有足夠的瑕疵樣本可以學。為了克服這個痛點，我的專案導入了無監督異常檢測（Unsupervised Anomaly Detection）思維。對於一維的電性數據，我利用 孤立森林（Isolation Forest） 與 One-Class SVM 建構雙重防線，不需要標籤就能把偏離常態分佈的髒資料和異常晶粒圈選出來。而對於二維的晶圓點位圖空間特徵，我實作了基於 PyTorch 的 卷積自動編碼器（ConvAutoEncoder）。我們只用大量正常的晶圓數據去訓練它，當產線出現從未見過的未知全新缺陷時，模型會因為無法順利重構，而產生極高的 重建誤差（Reconstruction Error）。這個設計能幫產線即時阻擋未知風險，也是柔性測試集成工程師不可或缺的AI工程能力。
